# Seniority EDA — pre-modelling checks

In [ ]:
import sys, os, django
sys.path.insert(0, os.path.abspath('..'))
os.environ.setdefault('DJANGO_SETTINGS_MODULE', 'config.settings.local')
django.setup()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from apps.intelligence.models import Job

df = pd.DataFrame(
    Job.objects.values('id', 'title', 'description', 'source', 'seniority')
)
print(df.shape)
df.head(3)

## 1. Class distribution

In [ ]:
dist = df['seniority'].value_counts()
print(dist)
print('\nSmallest / largest ratio:', round(dist.min() / dist.max(), 3))
dist.plot(kind='bar', title='Seniority class distribution')
plt.tight_layout()

**What to look for:**
- Smallest/largest ratio below ~0.15 → severe imbalance. A model trained on this will predict the majority class almost always and still get high accuracy. Fix: merge thin classes (e.g. `principal`+`lead`+`manager` → `senior-plus`), or use `class_weight='balanced'` in sklearn.
- A class with fewer than ~30 examples → not enough signal. Merge or drop it from training.
- `not_stated` being the largest class → decide now whether to treat it as a real class or exclude it. Including it usually hurts recall on the classes you actually care about.

## 2. Missing / empty labels

In [ ]:
blank = (df['seniority'] == '').sum()
null  = df['seniority'].isna().sum()
print(f'Blank seniority : {blank}')
print(f'Null  seniority : {null}')
print(f'Empty description: {(df["description"].str.strip() == "").sum()}')

**What to look for:**
- Any blank/null seniority → these are unlabelled and must be excluded from training (or sent to a separate inference set). If there are many, re-run `backfill_seniority`.
- Empty description → the classifier will have no text signal for these rows. Either drop them or fall back to title-only features. More than ~10% empty is a problem.

## 3. Description length by class

In [ ]:
df['desc_len'] = df['description'].str.split().str.len().fillna(0).astype(int)

print(df.groupby('seniority')['desc_len'].describe()[['min','50%','max']].to_string())

df.boxplot(column='desc_len', by='seniority', figsize=(10, 4))
plt.suptitle('')
plt.title('Description word count by seniority')
plt.tight_layout()

**What to look for:**
- Median below ~50 words for any class → descriptions are too short to carry useful signal; the classifier will lean heavily on the title. Consider concatenating title + description as your feature.
- One class has systematically much longer descriptions than others → the model may learn length as a proxy for the label, which won't generalise. Check whether that class comes from one source (see cross-tab below).
- High variance (IQR very wide) → noisy text quality; TF-IDF with a max_features cap will help.

## 4. Source × class cross-tab

In [ ]:
ct = pd.crosstab(df['source'], df['seniority'], margins=True)
print(ct)

# Proportions (exclude the All margin row/col)
ct_pct = pd.crosstab(df['source'], df['seniority'], normalize='index').round(2)
print('\nRow proportions:')
print(ct_pct)

**What to look for:**
- One source dominates a class (e.g. 90%+ of `intern` comes from `arbeitnow`) → the model will learn source as a shortcut. At inference time on a new source it will fail. Fix: add source as an explicit feature so the model can use it honestly, or stratify your train/test split by source.
- A source has near-zero representation in a class → if that source appears in your test set, you'll get misleading eval numbers. Make sure your test split is stratified by both source and class.
- One source has far fewer total jobs → if it ends up only in test, the model has never seen its writing style. Ensure every source appears in training.